# 개선 가설 실험 1단계 (H1, H2, H3): 독감, 날씨, 전력, 교통

**목적.** `docs/improvement_hypotheses.md`의 가설 가운데 H1, H2, H3을 확인한다. ETT 4종과 환율은 CPU로 이미 돌렸고, 이 노트북은 나머지 네 데이터를 돌린다.

| 이름 | 내용 | 가설 |
|---|---|---|
| DLinear | 기준. 논문값이 나오는 2022-06 코드와 설정, 시드 2021~2025 | |
| DLinear+sub | DLinear 앞에서 마지막 값을 빼고 뒤에서 다시 더한다. 설정은 기준과 같다 | H1 |
| Ridge, Ridge+sub | 학습 없이 수식으로 푼 해. 규제 세기는 검증 구간으로 고른다. `+sub`는 마지막 값을 뺀 것 | H2 |
| seed = ens | 시드 5개의 가중치를 평균한 모델 (선형 모델이라 예측 평균과 같다) | H3 |

**실행.** 런타임 **GPU → T4** → **모두 실행**. Google Drive 연결을 허용하면 결과가 Drive의 `dlinear_hyp/results.csv`에 쌓인다. 런타임이 끊겨도 다시 **모두 실행**하면 끝난 칸은 건너뛴다.

**예상 시간 (T4, 추정).** 독감 5분, 날씨 1~2시간, 전력 4~6시간, 교통 40~60시간. 가벼운 데이터부터 돈다. 교통은 논문값을 내려면 30 epoch까지 학습해야 해서 오래 걸린다. 중간에 멈춰도 그때까지의 결과는 남으니, 앞의 세 데이터가 끝나면 결과를 먼저 보내도 된다.

**규칙.** 시드 5개의 평균과 표준편차로 비교한다. 테스트 숫자를 보고 설정을 고르지 않는다. 릿지의 규제 세기만 고르는 값이고, 검증 구간으로 고른다.

In [ ]:
# 0. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. 코드와 데이터 준비, 결과 저장 위치
import os, subprocess
os.chdir('/content')
subprocess.run('pip -q install gdown', shell=True)
if not os.path.exists('LTSF-Linear'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cure-lab/LTSF-Linear.git'], check=True)
os.makedirs('/content/data', exist_ok=True)
if not os.path.exists('/content/data/traffic.csv'):
    subprocess.run('gdown -q --folder '
                   'https://drive.google.com/drive/folders/1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy -O /content/raw_data && '
                   'find /content/raw_data -name "*.csv" -exec cp -n {} /content/data/ \\;', shell=True, check=True)
# 2022-06 코드(논문의 DLinear 숫자가 나오는 코드)와 2022-08 코드를 옆 폴더에 꺼낸다. 코드는 고치지 않는다.
for name, commit in [('june', '04ffd8d'), ('aug', '7f7a25b')]:
    d = f'/content/{name}'
    if not os.path.exists(d):
        r = subprocess.run(['git', '-C', '/content/LTSF-Linear', 'worktree', 'add', '-q', d, commit])
        if r.returncode != 0:
            subprocess.run(['git', '-C', '/content/LTSF-Linear', 'fetch', '-q', '--unshallow'])
            subprocess.run(['git', '-C', '/content/LTSF-Linear', 'worktree', 'add', '-q', d, commit], check=True)
    os.makedirs(f'{d}/dataset', exist_ok=True)
    for f in os.listdir('/content/data'):
        if not os.path.exists(f'{d}/dataset/{f}'):
            os.symlink(f'/content/data/{f}', f'{d}/dataset/{f}')
!git -C /content/june log --oneline -1
!git -C /content/aug log --oneline -1
!ls /content/data
OUTDIR = '/content/hyp_out'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/dlinear_hyp'
except Exception as e:
    print('Drive 연결 안 됨, /content에 저장:', e)
os.makedirs(OUTDIR, exist_ok=True)
print('결과 폴더:', OUTDIR)

In [ ]:
%%writefile /content/lab.py
"""DLinear 추가 실험 도우미.

공식 스크립트(scripts/EXP-*/...sh)를 '가짜 python'으로 한 번 돌려서 실제 실행될 명령 목록만 뽑고,
그 명령을 하나씩 실행한다. 이렇게 하면
  - 하이퍼파라미터를 손으로 옮겨 적지 않고 공식 스크립트 값을 그대로 쓰고
  - 실험마다 진행 상황이 한 줄씩 찍히고
  - 중간에 끊겨도 다시 실행하면 끝난 실험은 건너뛴다.
"""
import json, os, re, shutil, subprocess, sys, time

import pandas as pd

ROOT = '/content/LTSF-Linear'
_SHIM = '/content/_shim'
_WORK = '/content/_collect'

DATASETS = {'etth1.csv': 'ETTh1', 'etth2.csv': 'ETTh2', 'ettm1.csv': 'ETTm1', 'ettm2.csv': 'ETTm2',
            'electricity.csv': 'Electricity', 'exchange_rate.csv': 'Exchange', 'traffic.csv': 'Traffic',
            'weather.csv': 'Weather', 'national_illness.csv': 'ILI'}


def commands(script, edits=()):
    """공식 스크립트에서 실행될 python 명령 목록을 뽑는다. edits = [(찾을 문자열, 바꿀 문자열), ...]"""
    os.makedirs(_SHIM, exist_ok=True)
    with open(f'{_SHIM}/python', 'w') as f:
        f.write('#!/usr/bin/env python3\n'
                'import json, os, sys\n'
                'try:\n    log = os.readlink("/proc/self/fd/1")\nexcept OSError:\n    log = ""\n'
                'with open(os.environ["CMD_FILE"], "a") as f:\n'
                '    f.write(json.dumps({"argv": sys.argv[1:], "log": log}) + "\\n")\n')
    os.chmod(f'{_SHIM}/python', 0o755)

    shutil.rmtree(_WORK, ignore_errors=True)
    for d in ['logs/LongForecasting/univariate', 'logs/LookBackWindow']:  # 스크립트가 안 만드는 폴더도 미리
        os.makedirs(f'{_WORK}/{d}')
    src = open(os.path.join(ROOT, script)).read()
    for a, b in edits:
        assert a in src, f'{a!r} 가 {script} 에 없음'
        src = src.replace(a, b)
    open(f'{_WORK}/run.sh', 'w').write(src)
    env = dict(os.environ, PATH=f'{_SHIM}:{os.environ["PATH"]}', CMD_FILE=f'{_WORK}/cmds.jsonl')
    subprocess.run(['bash', 'run.sh'], cwd=_WORK, env=env, check=True, stdout=subprocess.DEVNULL)
    out = [json.loads(l) for l in open(f'{_WORK}/cmds.jsonl')]
    for c in out:
        c['log'] = os.path.basename(c['log']) if c['log'].startswith(_WORK) else ''
    return out


def arg(c, *names, default=None):
    a = c['argv']
    for i, x in enumerate(a[:-1]):
        if x in names:
            return a[i + 1]
    return default


def set_arg(c, name, value):
    a = c['argv']
    if name in a:
        a[a.index(name) + 1] = str(value)
    else:
        a += [name, str(value)]
    return c


def info(c):
    return dict(model=arg(c, '--model'), dataset=DATASETS[arg(c, '--data_path').lower()],
                features=arg(c, '--features', '--feature', default='M'),
                seq_len=int(arg(c, '--seq_len')), pred_len=int(arg(c, '--pred_len')),
                individual='--individual' in c['argv'])


def parse_log(path):
    """로그에서 (testing 설정, mse, mae)를 순서대로 뽑는다. --itr N이면 N개."""
    if not os.path.exists(path):
        return []
    text = open(path, errors='ignore').read()
    out = []
    for m in re.finditer(r'>>>>>>>testing : (\S+?)<<<.*?mse:([-\d.eE+]+), mae:([-\d.eE+]+)', text, re.S):
        out.append((m.group(1), float(m.group(2)), float(m.group(3))))
    return out


def run(cmds, tag):
    """명령을 차례로 실행. 로그는 logs/<tag>/ 에 저장, 이미 끝난 실험은 건너뜀."""
    logdir = os.path.join(ROOT, 'logs', tag)
    os.makedirs(logdir, exist_ok=True)
    t0 = time.time()
    for i, c in enumerate(cmds, 1):
        c['logpath'] = os.path.join(logdir, c.get('name') or c['log'] or f'run{i}.log')
        d = info(c)
        label = (f"{d['dataset']} {d['features']} L={d['seq_len']} T={d['pred_len']}"
                 + (' individual' if d['individual'] else ''))
        need = int(arg(c, '--itr', default=1))
        if len(parse_log(c['logpath'])) >= need:
            print(f'[{i}/{len(cmds)}] {label}: 이미 완료, 건너뜀', flush=True)
            continue
        t = time.time()
        with open(c['logpath'], 'w') as f:
            r = subprocess.run([sys.executable] + c['argv'], cwd=ROOT, stdout=f, stderr=subprocess.STDOUT)
        subprocess.run('rm -f results/*/pred.npy', shell=True, cwd=ROOT)  # 예측 배열은 안 쓰므로 디스크 절약
        res = parse_log(c['logpath'])
        if r.returncode != 0 or len(res) < need:
            tail = open(c['logpath'], errors='ignore').read()[-600:]
            print(f'[{i}/{len(cmds)}] {label}: 실패 (exit {r.returncode})\n{tail}', flush=True)
            continue
        mses = ', '.join(f'{m:.4f}' for _, m, _ in res)
        print(f'[{i}/{len(cmds)}] {label}: mse {mses}  ({time.time() - t:.0f}s, 누적 {(time.time() - t0) / 60:.0f}분)',
              flush=True)


def collect(cmds):
    rows = []
    for c in cmds:
        for k, (setting, mse, mae) in enumerate(parse_log(c.get('logpath', ''))):
            rows.append(dict(info(c), run=k, mse=mse, mae=mae))
    return pd.DataFrame(rows)

In [ ]:
%%writefile /content/hyp_exp.py
"""One run of the improvement-hypothesis experiments, on an unmodified LTSF-Linear checkout.

Usage:
  python hyp_exp.py <repo_dir> key=value ... -- <the official run_longExp.py arguments>

Keys
  mode     train | ens | ridge
  seed     replaces the repository's hard-coded fix_seed = 2021           (train)
  variant  base | sublast     sublast = subtract the last input value before the model and
                              add it back after (H1 for DLinear; for ridge, NLinear-style)
  keep     file to copy the best checkpoint to                            (train)
  ckpts    comma-separated checkpoints whose weights are averaged         (ens, H3)

What is changed and what is not
  - run_longExp.py is executed as it is, except for the seed line and, for ens, --is_training 0.
  - Training, early stopping, learning-rate schedule and data loading are the repository's.
  - The test step is replaced by one that adds up the errors batch by batch in float64. It gives
    the repository's numbers (same loader, same drop_last) without holding all predictions in memory.
  - ens: for a model that is linear in its weights, averaging weights equals averaging predictions.
  - ridge: closed-form ridge regression with an intercept, one weight shared by all variables,
    on the same scaled splits. The penalty is lam * (number of training windows) * ||W||^2, the
    intercept is not penalised, and lam is chosen on the validation split. Test error is reported
    on the same windows the repository's test loader keeps (drop_last), and on all windows.

Prints
  train/ens:  the repository's log, ending with   mse:<mse>, mae:<mae>
  ridge:      RIDGE,<lam>,<val_mse>,<test_mse>,<test_mae>,<test_mse_all>   per lam
              BEST,<lam>,<val_mse>,<test_mse>,<test_mae>,<test_mse_all>
"""
import glob
import os
import shutil
import sys

repo = sys.argv[1]
cut = sys.argv.index('--')
kv = dict(a.split('=', 1) for a in sys.argv[2:cut])
OFF = sys.argv[cut + 1:]
MODE = kv.get('mode', 'train')
SEED = int(kv.get('seed', 2021))
VAR = kv.get('variant', 'base')
CKPTS = [c for c in kv.get('ckpts', '').split(',') if c]
LAMS = [0, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]

os.chdir(repo)
sys.path.insert(0, repo)

import numpy as np
import torch

if not hasattr(np, 'Inf'):  # NumPy 2 removed np.Inf, which the repository still uses
    np.Inf = np.inf

if VAR == 'sublast' and MODE != 'ridge':
    import models.DLinear as D
    _forward = D.Model.forward

    def forward(self, x):
        last = x[:, -1:, :].detach()
        return _forward(self, x - last) + last
    D.Model.forward = forward

import exp.exp_main as em


def test(self, setting, test=0):
    _, loader = self._get_data(flag='test')
    if CKPTS:
        states = [torch.load(c, map_location=self.device) for c in CKPTS]
        self.model.load_state_dict({k: sum(s[k] for s in states) / len(states) for k in states[0]})
    elif test:
        self.model.load_state_dict(torch.load(os.path.join('./checkpoints/' + setting, 'checkpoint.pth')))
    P = self.args.pred_len
    f = -1 if self.args.features == 'MS' else 0
    se = ae = n = 0
    self.model.eval()
    with torch.no_grad():
        for bx, by, _, _ in loader:
            out = self.model(bx.float().to(self.device))
            d = (out[:, -P:, f:] - by.float().to(self.device)[:, -P:, f:]).double()
            se += float((d ** 2).sum())
            ae += float(d.abs().sum())
            n += d.numel()
    print('mse:{}, mae:{}'.format(se / n, ae / n))


em.Exp_Main.test = test

src = open('run_longExp.py').read()
assert 'fix_seed = 2021' in src
src = src.replace('fix_seed = 2021', 'fix_seed = %d' % SEED)

if MODE in ('train', 'ens'):
    if MODE == 'ens':
        OFF[OFF.index('--is_training') + 1] = '0'
    sys.argv = ['run_longExp.py'] + OFF
    exec(compile(src, 'run_longExp.py', 'exec'), {'__name__': '__main__'})
    mid = OFF[OFF.index('--model_id') + 1]
    for d in glob.glob('checkpoints/%s_*' % mid):
        if MODE == 'train' and 'keep' in kv:
            os.makedirs(os.path.dirname(kv['keep']), exist_ok=True)
            shutil.copy(os.path.join(d, 'checkpoint.pth'), kv['keep'])
        shutil.rmtree(d)
    sys.exit(0)

# ---------------------------------------------------------------- ridge (H2)
sys.argv = ['run_longExp.py'] + OFF
ns = {'__name__': 'head'}
exec(compile(src.split('Exp = Exp_Main')[0], 'run_longExp.py', 'exec'), ns)
args = ns['args']
from data_provider.data_factory import data_provider

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
L, T = args.seq_len, args.pred_len
SUB = VAR == 'sublast'


def windows(ds):
    X = torch.tensor(ds.data_x, dtype=torch.float64, device=dev)
    N = len(ds)
    for c in range(X.shape[1]):
        s = X[:, c]
        xw, yw = s.unfold(0, L, 1)[:N], s.unfold(0, T, 1)[L:L + N]
        last = xw[:, -1:] if SUB else 0
        A = torch.cat([xw - last, torch.ones(N, 1, dtype=torch.float64, device=dev)], 1)
        yield A, yw, last


def errors(ds, W, keep=None):
    se = ae = n = 0
    for A, yw, last in windows(ds):
        d = (A @ W + last - yw)[:keep]
        se += float((d ** 2).sum())
        ae += float(d.abs().sum())
        n += d.numel()
    return se / n, ae / n


tr, va, te = (data_provider(args, f)[0] for f in ('train', 'val', 'test'))
G = torch.zeros(L + 1, L + 1, dtype=torch.float64, device=dev)
B = torch.zeros(L + 1, T, dtype=torch.float64, device=dev)
n = 0
for A, yw, last in windows(tr):
    G += A.T @ A
    B += A.T @ (yw - last)
    n += A.shape[0]
pen = torch.eye(L + 1, dtype=torch.float64, device=dev)
pen[-1, -1] = 0
keep = len(te) // args.batch_size * args.batch_size
best = None
for lam in LAMS:
    W = torch.linalg.pinv(G, hermitian=True) @ B if lam == 0 else torch.linalg.solve(G + lam * n * pen, B)
    row = (lam, errors(va, W)[0]) + errors(te, W, keep) + (errors(te, W)[0],)
    print('RIDGE,' + ','.join(str(x) for x in row))
    if best is None or row[1] < best[1]:
        best = row
print('BEST,' + ','.join(str(x) for x in best))

In [ ]:
%%writefile /content/hyp_run.py
"""Stage 1 of the improvement hypotheses (H1, H2, H3): builds the job list and runs it.

Usage:
  python hyp_run.py <june_repo> <aug_repo> <out_dir> <datasets, comma-separated> [workers] [num_workers]

  june_repo   LTSF-Linear at 04ffd8d (the code that gives the paper's DLinear numbers)
  aug_repo    LTSF-Linear at 7f7a25b (the first code with NLinear)
  datasets    ETTh1,ETTh2,ETTm1,ETTm2,exchange_rate,ili,weather,electricity,traffic

Per dataset and forecast length:
  Ridge, Ridge+sub          closed-form solution, plain and with the last value subtracted   (H2)
  DLinear                   baseline, seeds 2021-2025
  DLinear+sub               DLinear with the last value subtracted, same settings            (H1)
  NLinear                   baseline, seeds 2021-2025 (only where its settings were reproduced)
  seed = ens                the five seeds' weights averaged                                 (H3)

Settings come from the repositories' own scripts, with the changes that reproduce the paper:
  DLinear  ILI learning rate 0.01, ETTm2 720 learning rate 0.01, Traffic up to 30 epochs
  NLinear  ETTh2 learning rate 0.005, ETTm2 720 learning rate 0.01

Finished jobs are in <out_dir>/results.csv and are skipped when the script is run again.
"""
import csv
import os
import re
import subprocess
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import lab

JUNE, AUG, OUT = sys.argv[1], sys.argv[2], sys.argv[3]
DATASETS = sys.argv[4].split(',')
WORKERS = int(sys.argv[5]) if len(sys.argv) > 5 else 1
NUM_WORKERS = sys.argv[6] if len(sys.argv) > 6 else '2'
SEEDS = [2021, 2022, 2023, 2024, 2025]
NLINEAR_DATA = ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'exchange_rate']  # cells reproduced with the Aug code
COLS = ['model', 'data', 'pred_len', 'seed', 'lr', 'batch', 'lam', 'mse', 'mae', 'mse_all', 'val',
        'epochs_run', 'best_epoch', 'seconds']
RES = os.path.join(OUT, 'results.csv')
os.makedirs(os.path.join(OUT, 'logs'), exist_ok=True)
os.makedirs(os.path.join(OUT, 'ckpt'), exist_ok=True)
lab._SHIM, lab._WORK = os.path.join(OUT, '_shim'), os.path.join(OUT, '_collect')
lock = threading.Lock()


def done():
    if not os.path.exists(RES):
        return set()
    return {(r['model'], r['data'], r['pred_len'], r['seed']) for r in csv.DictReader(open(RES))}


def save(**row):
    with lock:
        new = not os.path.exists(RES)
        with open(RES, 'a', newline='') as f:
            w = csv.DictWriter(f, COLS)
            if new:
                w.writeheader()
            w.writerow({c: row.get(c, '') for c in COLS})


def official(repo, sub, name, model):
    lab.ROOT = repo
    edits = [('model_name=DLinear', 'model_name=' + model)] if model != 'DLinear' else []
    cmds = lab.commands('scripts/EXP-LongForecasting/%s/%s.sh' % (sub, name.lower()), edits)
    assert len(cmds) == 4, (name, len(cmds))
    for c in cmds:
        p = lab.arg(c, '--pred_len')
        lab.set_arg(c, '--num_workers', NUM_WORKERS)
        if model == 'DLinear':
            if name == 'ili' or (name == 'ETTm2' and p == '720'):
                lab.set_arg(c, '--learning_rate', '0.01')
            if name == 'traffic':
                lab.set_arg(c, '--train_epochs', '30')
        else:
            if name == 'ETTh2':
                lab.set_arg(c, '--learning_rate', '0.005')
            if name == 'ETTm2' and p == '720':
                lab.set_arg(c, '--learning_rate', '0.01')
    return cmds


def run(repo, c, model, data, seed, kv):
    p = lab.arg(c, '--pred_len')
    key = (model, data, p, str(seed))
    if key in DONE:
        return
    tag = '%s_%s_%s_%s' % (model.replace('+', ''), data, p, seed)
    argv = c['argv'][c['argv'].index('run_longExp.py') + 1:]
    argv[argv.index('--model_id') + 1] = 'hyp_' + tag
    log = os.path.join(OUT, 'logs', tag + '.log')
    t = time.time()
    with open(log, 'w') as f:
        r = subprocess.run([sys.executable, '-u', os.path.join(HERE, 'hyp_exp.py'), repo]
                           + ['%s=%s' % x for x in kv.items()] + ['--'] + argv,
                           stdout=f, stderr=subprocess.STDOUT, env=dict(os.environ, OMP_NUM_THREADS='1'))
    s = open(log, errors='ignore').read()
    row = dict(model=model, data=data, pred_len=p, seed=seed, lr=lab.arg(c, '--learning_rate'),
               batch=lab.arg(c, '--batch_size'), seconds=round(time.time() - t))
    if kv.get('mode') == 'ridge':
        m = re.search(r'^BEST,(.*)$', s, re.M)
        if m:
            lam, val, mse, mae, mse_all = m.group(1).split(',')
            row.update(lam=lam, val=val, mse=mse, mae=mae, mse_all=mse_all, lr='', seed='-')
    else:
        m = re.search(r'^mse:([-\d.eE+]+), mae:([-\d.eE+]+)', s, re.M)
        if m:
            vali = [float(v) for v in re.findall(r'^Epoch: \d+, Steps.*Vali Loss: ([-\d.eE+]+)', s, re.M)]
            row.update(mse=m.group(1), mae=m.group(2))
            if vali:
                row.update(epochs_run=len(vali), best_epoch=vali.index(min(vali)) + 1, val=min(vali))
    if r.returncode != 0 or not m:
        print('FAILED %s\n%s' % (tag, s[-500:]), flush=True)
        return
    save(**row)
    print('%-34s mse %.4f  (%ds)' % (tag, float(row['mse']), row['seconds']), flush=True)


def ckpt(model, data, p, seed):
    return os.path.join(OUT, 'ckpt', '%s_%s_%s_%s.pth' % (model.replace('+', ''), data, p, seed))


for data in DATASETS:
    DONE = done()
    dl = official(JUNE, 'DLinear', data, 'DLinear')
    nl = official(AUG, 'Linear', data, 'NLinear') if data in NLINEAR_DATA else []
    jobs = []
    for c in dl:
        p = lab.arg(c, '--pred_len')
        jobs.append((JUNE, c, 'Ridge', data, '-', dict(mode='ridge')))
        jobs.append((JUNE, c, 'Ridge+sub', data, '-', dict(mode='ridge', variant='sublast')))
        for s in SEEDS:
            jobs.append((JUNE, c, 'DLinear', data, s, dict(mode='train', seed=s, keep=ckpt('DLinear', data, p, s))))
            jobs.append((JUNE, c, 'DLinear+sub', data, s,
                         dict(mode='train', seed=s, variant='sublast', keep=ckpt('DLinear+sub', data, p, s))))
    for c in nl:
        p = lab.arg(c, '--pred_len')
        for s in SEEDS:
            jobs.append((AUG, c, 'NLinear', data, s, dict(mode='train', seed=s, keep=ckpt('NLinear', data, p, s))))
    with ThreadPoolExecutor(WORKERS) as ex:
        list(ex.map(lambda j: run(*j), jobs))
    DONE = done()
    ens = []
    for repo, cmds, models in [(JUNE, dl, ['DLinear', 'DLinear+sub']), (AUG, nl, ['NLinear'])]:
        for c in cmds:
            p = lab.arg(c, '--pred_len')
            for m in models:
                files = [ckpt(m, data, p, s) for s in SEEDS]
                if all(os.path.exists(f) for f in files):
                    kv = dict(mode='ens', ckpts=','.join(files))
                    if m == 'DLinear+sub':
                        kv['variant'] = 'sublast'
                    ens.append((repo, c, m, data, 'ens', kv))
    with ThreadPoolExecutor(WORKERS) as ex:
        list(ex.map(lambda j: run(*j), ens))
    print('== %s done' % data, flush=True)

## 2. 실행
가벼운 데이터부터 돈다. 한 줄에 실행 하나씩 찍힌다. 일부만 돌리려면 데이터 이름을 줄이면 된다.

In [ ]:
DATA = 'ili,weather,electricity,traffic'
!python -u /content/hyp_run.py /content/june /content/aug {OUTDIR} {DATA} 1 2

## 3. 결과를 채팅에 보내 주세요
Drive의 `dlinear_hyp/results.csv` 파일을 그대로 첨부하면 된다. 일부만 끝났어도 괜찮다.

In [ ]:
import pandas as pd
r = pd.read_csv(OUTDIR + '/results.csv')
t = r[~r.seed.astype(str).isin(['ens', '-'])].groupby(['data', 'pred_len', 'model']).mse.agg(['mean', 'std', 'count']).round(4)
print(t.to_string())
print(r[r.seed.astype(str).isin(['ens', '-'])][['model', 'data', 'pred_len', 'seed', 'lam', 'mse']].to_string())